## Probing: can frozen encodings predict ISUP grade (PANDA)?

PANDA's target, `isup_grade`, is an ordinal 0-5 cancer grade derived from Gleason patterns. This notebook probes the **slide-level** mean-pooled encodings that `00_preparation/01_encodings/encode_panda.ipynb` saves (one vector per slide, averaged over its sampled patches) -- the natural granularity for a slide-level label.

This loads the shared, full-scale encodings from `/home/shared/data/panda/encodings/` rather than each participant's own (smaller) run, so slide-level probing here has a larger pool to draw from than a single `N_SLIDES = 100` run would give. The probe itself runs on a 500-slide subsample (`N_SAMPLES` below) -- plenty for cross-validated metrics, and fast enough to run in seconds rather than fitting logistic regression on all ~10k slides.

In [ ]:
# Install dependencies, then restart the kernel. This is only needed once per environment.
# !pip install -U -r /home/shared/helper/requirements.txt

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from pathlib import Path

from nbhelper import (
    pd,
    plt,
    tqdm,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from probing import latest_matching, plot_confusion_matrix, quadratic_weighted_kappa

### 1. Load slide-level encodings

`{model}_slide_*.csv`, indexed by `image_id`, feature columns plus `isup_grade` and `data_provider` already joined in by the encoding notebook. Loaded from the shared `/home/shared/data/panda/encodings/` folder rather than each participant's own run.

In [ ]:
panda_base_dir = Path("/home/shared/data/panda/")
encodings_dir = Path("/home/shared/data/panda/encodings")

MODEL_NAMES = ["uni2-h", "virchow2"]
slide_feats = {}
for model_name in MODEL_NAMES:
    csv_path = latest_matching(encodings_dir, f"{model_name}_slide_*.csv")
    df = pd.read_csv(csv_path, index_col=0)
    slide_feats[model_name] = df
    print(f"{model_name:10s} <- {csv_path.name}  ({len(df)} slides)")

slide_feats[MODEL_NAMES[0]]["isup_grade"].value_counts().sort_index()

### 2. Cross-validated linear probe

5-fold stratified CV (grade as the stratification key), multinomial logistic regression on standardized features. Reports plain accuracy/macro-F1 alongside **quadratic weighted kappa** -- PANDA's own competition metric, which credits near-miss predictions (grade 3 predicted as 4) more than far-off ones (grade 0 predicted as 5).

In [ ]:
N_SPLITS = 5
N_SAMPLES = 500  # random slide subsample -- keeps the CV fit fast (seconds, not minutes)

results = []
confusions = {}

for model_name in tqdm(MODEL_NAMES):
    print(f"Probing {model_name}...")

    df = slide_feats[model_name].sample(n=N_SAMPLES, random_state=42)
    feat_cols = [c for c in df.columns if c not in ("isup_grade", "data_provider")]

    X = df[feat_cols].values
    y = df["isup_grade"].values

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=0.5, max_iter=200, random_state=42),
    )
    y_pred = cross_val_predict(clf, X, y, cv=cv)

    results.append(
        {
            "encoder": model_name,
            "accuracy": accuracy_score(y, y_pred),
            "macro_f1": f1_score(y, y_pred, average="macro"),
            "quadratic_weighted_kappa": quadratic_weighted_kappa(y, y_pred),
        }
    )
    confusions[model_name] = confusion_matrix(y, y_pred, labels=sorted(df["isup_grade"].unique()))

results_df = pd.DataFrame(results).set_index("encoder")
results_df

### 3. Confusion matrix

Ordinal structure to look for: errors should cluster near the diagonal (adjacent grades), not scatter uniformly.

In [ ]:
fig, axs = plt.subplots(figsize=(7, 6))
model_name = "uni2-h"
grades = sorted(slide_feats[model_name]["isup_grade"].unique())
plot_confusion_matrix(
    confusions[model_name], grades, axs, title=f"{model_name} -- ISUP grade (5-fold CV)"
)
plt.tight_layout()
plt.show()

### 4. Does accuracy differ by data provider?

Radboud and Karolinska used different grading protocols historically (`explore_panda.ipynb`'s Gleason/ISUP cross-tab quirk) -- worth checking whether the probe is effectively better calibrated to one institution's labeling.

In [ ]:
provider_results = []
df = slide_feats[model_name].sample(n=N_SAMPLES, random_state=42)
feat_cols = [c for c in df.columns if c not in ("isup_grade", "data_provider")]
X = df[feat_cols].values
y = df["isup_grade"].values

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=1000, random_state=42))
y_pred = cross_val_predict(clf, X, y, cv=cv)

for provider, group in df.assign(y_pred=y_pred, y_true=y).groupby("data_provider"):
    print(f"{model_name:10s} -- {provider:10s} -- {len(group)} slides")
    provider_results.append(
        {
            "encoder": model_name,
            "data_provider": provider,
            "n_slides": len(group),
            "accuracy": accuracy_score(group["y_true"], group["y_pred"]),
            "quadratic_weighted_kappa": quadratic_weighted_kappa(group["y_true"], group["y_pred"]),
        }
    )

pd.DataFrame(provider_results).set_index(["encoder", "data_provider"])

### Takeaways

- Quadratic weighted kappa above ~0.6-0.7 would be competitive with early PANDA-challenge baselines -- treat this as a rough sanity check, not a leaderboard submission (level-0 tiling without MPP calibration, mean-pooled rather than attention-pooled slide features).
- A confusion matrix concentrated near the diagonal, even with modest raw accuracy, means the encoder is picking up *some* grade-monotonic signal.
- A per-provider accuracy gap is the expected symptom of PANDA's known cross-institution labeling inconsistency, not necessarily an encoder weakness.